# Setup Dataset e Preparazione per ResNet

Configurazione per scaricare e preparare il dataset Fruits Classification da Kaggle

In [ ]:
import os
import sys
import glob
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms, models

from tqdm import tqdm

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT = Path('..')
BATCH_SIZE = 32
IMG_SIZE = 224

# ===== TRASFORMAZIONI =====
NORMALIZE = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)

TRAIN_TRANSFORMS = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    NORMALIZE
])

VAL_TEST_TRANSFORMS = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    NORMALIZE
])

## 2. Verifica Dataset Scaricato

Il dataset è già pre-diviso in:
- **train/** (~1900-1940 immagini per classe)
- **valid/** (~39-40 immagini per classe)
- **test/** (~19-20 immagini per classe)

Totale: ~2000 immagini per classe (perfettamente bilanciato)

In [ ]:
dataset_path = ROOT / "data" / "fruits-classification"

# Verifica che il dataset sia stato scaricato
if dataset_path.exists():
    print(f"✓ Dataset trovato in: {dataset_path}")
    for split in ['train', 'valid', 'test']:
        split_path = dataset_path / split
        print(f"\n{split.upper()}:")
        for class_name in sorted(os.listdir(split_path)):
            class_folder = split_path / class_name
            if class_folder.is_dir():
                num_images = len(list(class_folder.glob("*.jpeg")))
                print(f"  {class_name}: {num_images} immagini")
else:
    print(f"⚠ Dataset non trovato in {dataset_path}")

## 3. Caricamento Percorsi Immagini da Cartelle Pre-divise

Poiché il dataset è già diviso in train/valid/test, leggiamo i percorsi direttamente dalle cartelle senza fare uno split manuale.

In [ ]:
def load_image_paths(split_path):
    """Carica i percorsi delle immagini da una cartella split"""
    paths = []
    labels = []
    for class_name in sorted(os.listdir(split_path)):
        class_folder = split_path / class_name
        if class_folder.is_dir():
            for img_file in glob.glob(str(class_folder / "*.jpeg")):
                paths.append(img_file)
                labels.append(class_name)
    return paths, labels

# Carica i percorsi da train, valid, test
train_paths, train_labels = load_image_paths(dataset_path / 'train')
val_paths, val_labels = load_image_paths(dataset_path / 'valid')
test_paths, test_labels = load_image_paths(dataset_path / 'test')

print(f"Totale Train: {len(train_paths)} immagini")
print(f"Totale Validation: {len(val_paths)} immagini")
print(f"Totale Test: {len(test_paths)} immagini")

## 4. DataLoader - Preparazione dei Dati per la Rete Neurale

I **DataLoader** sono strumenti che si occupano di:

1. **Creazione di batch:** Le immagini non vengono caricate tutte insieme, ma suddivise in pacchetti (batch) per evitare problemi di memoria
2. **Mescolamento:** Le immagini vengono mescolate casualmente durante il training, così la rete neurale apprende realmente e non impara solo l'ordine
3. **Trasformazioni:** Conversione delle immagini in tensori matematici, normalizzazione, augmentation, ecc.

Creeremo tre DataLoader (uno per Train, uno per Validation, uno per Test) che applicheranno automaticamente le trasformazioni.

In [ ]:
class FruitsDataset(torch.utils.data.Dataset):
    def __init__(self, image_paths, labels, transforms=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transforms = transforms
        self.classes = sorted(list(set(labels)))
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        label_idx = self.class_to_idx[label]
        
        img = Image.open(img_path).convert('RGB')
        
        if self.transforms:
            img = self.transforms(img)
        
        return img, label_idx

train_dataset = FruitsDataset(train_paths, train_labels, transforms=TRAIN_TRANSFORMS)
val_dataset = FruitsDataset(val_paths, val_labels, transforms=VAL_TEST_TRANSFORMS)
test_dataset = FruitsDataset(test_paths, test_labels, transforms=VAL_TEST_TRANSFORMS)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

print("✓ DataLoader creati")
print(f"Train: {len(train_loader)} batch")
print(f"Validation: {len(val_loader)} batch")
print(f"Test: {len(test_loader)} batch")